# Building Agentic AI Systems — ISMB Tutorial (SOLUTIONS)

A 1-hour hands-on introduction to agentic AI workflows for bioinformaticians, building on the *Module Guide* reference document.

## How to use this notebook

1. Run the **Setup** section once, before the tutorial starts.
2. During the live session, we'll work through sections **§1–§6** together.
3. Cells marked `# TODO` are short exercises (2–3 minutes each).
4. **§7 (MCP)** is a bonus section — covered live if we have time, otherwise yours to run on your own.

## Running example

Throughout the tutorial we build a single agent that extracts structured information from PubMed abstracts. Each section adds a capability: tool use → structured outputs → context → multi-agent.

---
## Setup — run this once

Install dependencies, set your **OpenRouter** API key, configure the model, and load the cached PubMed abstracts. Re-running these cells is safe.

> **Why OpenRouter?** One key, 300+ models, OpenAI-compatible API. You can swap models by changing one variable (`MODEL_SLUG`).
>
> **Tip.** Run this section the night before the tutorial to avoid WiFi pain in the conference room.

In [31]:
# 1. Install dependencies (quiet; safe to re-run)
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
# Jupyter / Cursor already have an event loop running; nest_asyncio lets
# PydanticAI's run_sync coexist with it instead of erroring out.
import nest_asyncio
nest_asyncio.apply()
print('✓ nest_asyncio applied — run_sync will work in this notebook')

✓ nest_asyncio applied — run_sync will work in this notebook


In [33]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = None    # e.g. 'sk-or-v1-...'

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

print('✓ API key set')

✓ API key set


In [34]:
# 3. Configure the model — used by every agent in this notebook.
# OpenRouter is OpenAI-compatible, so we use PydanticAI's OpenAI model
# class directly and just point it at OpenRouter's base URL.
# Change MODEL_SLUG to any OpenRouter model you have access to.
import httpx
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# ── Jupyter-specific workaround #1 ────────────────────────────────
# Jupyter already has an asyncio event loop running for UI events.
# PydanticAI's run_sync() tries to start its own — without this patch
# you'd get: RuntimeError: This event loop is already running.
import nest_asyncio
nest_asyncio.apply()

# ── Jupyter-specific workaround #2 ────────────────────────────────
# When `async with agent.run_mcp_servers()` exits under nest_asyncio,
# the cleanup occasionally closes the httpx client that OpenAIProvider
# shares across every agent built from this MODEL. Subsequent agent
# calls would then fail with: 'Cannot send a request, as the client
# has been closed.' Owning the httpx.AsyncClient ourselves prevents
# PydanticAI from closing it. (Plain `python script.py` doesn't need
# this — it's strictly a notebook lifecycle issue.)
HTTPX_CLIENT = httpx.AsyncClient(timeout=60)

MODEL_SLUG = 'anthropic/claude-haiku-4.5'   # try also: openai/gpt-5-mini, google/gemini-2.5-flash

MODEL = OpenAIChatModel(
    MODEL_SLUG,
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=HTTPX_CLIENT,
    ),
)

print(f'✓ Model configured: {MODEL_SLUG}')

✓ Model configured: anthropic/claude-haiku-4.5


In [35]:
# 4. Cached PubMed abstracts (used in place of live PubMed during the tutorial)
CACHED_ABSTRACTS = [
    {
        "pmid": "38421567",
        "title": "Deep learning identifies novel cardiac amyloidosis biomarkers in plasma proteomics",
        "year": 2024,
        "abstract": (
            "Cardiac amyloidosis is underdiagnosed; we trained a transformer-based "
            "model on 2,847 plasma proteomic profiles from Homo sapiens patients "
            "with confirmed transthyretin amyloidosis and matched controls. The "
            "model achieved 0.94 AUROC on a held-out test set and identified 17 "
            "previously unreported biomarker candidates, three of which validated "
            "in an independent cohort of 412 patients."
        ),
    },
    {
        "pmid": "39012245",
        "title": "CRISPR-Cas9 screen reveals zebrafish heart regeneration regulators",
        "year": 2024,
        "abstract": (
            "We performed a genome-wide CRISPR-Cas9 knockout screen in Danio rerio "
            "embryos to identify genes required for cardiac regeneration after "
            "ventricular resection. Screening 19,432 protein-coding genes, we "
            "identified 47 regeneration-essential loci with a false discovery rate "
            "below 5%. Loss-of-function of nrg1 reduced regeneration by 68%."
        ),
    },
    {
        "pmid": "37889123",
        "title": "Gut microbiome composition predicts inflammatory bowel disease relapse",
        "year": 2023,
        "abstract": (
            "Using shotgun metagenomic sequencing of stool samples from 612 IBD "
            "patients in remission, we built a random-forest classifier predicting "
            "relapse within 6 months. The model reached 0.81 AUROC, outperforming "
            "C-reactive protein (0.62). Faecalibacterium prausnitzii abundance was "
            "the strongest single predictor."
        ),
    },
    {
        "pmid": "40123987",
        "title": "Single-cell RNA-seq reveals tumour microenvironment in NSCLC",
        "year": 2025,
        "abstract": (
            "We profiled 187,433 single cells from 41 non-small-cell lung cancer "
            "(Homo sapiens) tumours using droplet-based scRNA-seq. Unsupervised "
            "clustering identified 22 distinct cell populations including a novel "
            "exhausted T-cell subtype expressing TOX and TIGIT. The exhausted "
            "subtype's abundance correlated with response to anti-PD-1 therapy "
            "(odds ratio 3.4)."
        ),
    },
    {
        "pmid": "39765432",
        "title": "AlphaFold-Multimer with diffusion refinement improves complex prediction",
        "year": 2025,
        "abstract": (
            "We extended AlphaFold-Multimer with a diffusion-based structural "
            "refinement module trained on 14,200 cryo-EM complexes. On the CASP16 "
            "benchmark, our method improved DockQ score by 18% over the baseline. "
            "The improvement was largest for antibody-antigen complexes (32% "
            "DockQ gain)."
        ),
    },
]
print(f"Loaded {len(CACHED_ABSTRACTS)} cached PubMed abstracts.")

# Also persist to disk so the MCP server in §7 can read the same data.
import json
from pathlib import Path
Path('abstracts.json').write_text(json.dumps(CACHED_ABSTRACTS, indent=2))
print('Wrote abstracts.json')

Loaded 5 cached PubMed abstracts.
Wrote abstracts.json


In [36]:
# 5. Sanity check — one round-trip to the model
_sanity_agent = Agent(MODEL)
_result = _sanity_agent.run_sync("Reply with the single word 'ready'.")
print('Model says:', _result.output)

Model says: ready


---
## §1  Chat vs Agent

A **plain LLM call** has no memory of the world after its training cutoff and no tools. An **agent** has a loop: it can call tools, see the results, and reason over them.

The simplest demonstration: ask for today's date.

In [37]:
# (a) Plain agent — no tools
from pydantic_ai import Agent

plain_agent = Agent(MODEL)
result = plain_agent.run_sync('What is today\'s date?')
print('Without tools:', result.output)

Without tools: I don't have access to real-time information, so I can't tell you today's specific date. 

To find out today's date, you can:
- Check your device (phone, computer, etc.)
- Search "today's date" online
- Ask your voice assistant

Is there something specific you'd like help with regarding a date?


In [38]:
# (b) Agent with one tool
from datetime import date

tool_agent = Agent(MODEL)

@tool_agent.tool_plain
def get_today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return date.today().isoformat()

result = tool_agent.run_sync('What is today\'s date?')
print('With tool:   ', result.output)

With tool:    Today's date is **May 5, 2026** (2026-05-05).


**What just happened?**

The first agent had no way to know the current date. The second agent had a tool — `get_today()` — and the LLM decided to call it.

Critically, the LLM doesn't execute the function itself. It returns a *tool-call request*, the framework runs the function, the result is appended to the conversation, and the LLM then produces its final answer.

That's the agent loop. Everything else builds on it.

---
## §2  The agent loop — inspecting the trace

The 5-step loop:

1. LLM receives prompt + list of available tools
2. LLM outputs a structured tool-call request
3. Framework executes the function
4. Return value is appended to the conversation
5. LLM generates the final response (possibly calling more tools first)

Let's inspect a multi-step trace.

In [39]:
result = tool_agent.run_sync(
    "What is today's date and what year is it?"
)

print(f'Final answer: {result.output}\n')
print('Trace (each ModelMessage in the conversation):')
for i, msg in enumerate(result.all_messages()):
    print(f'\n[{i}] {type(msg).__name__}')
    for part in getattr(msg, "parts", []):
        kind = type(part).__name__
        snippet = repr(part)[:200]
        print(f'    └─ {kind}: {snippet}')

Final answer: Today's date is **May 5, 2026** (2026-05-05), and the year is **2026**.

Trace (each ModelMessage in the conversation):

[0] ModelRequest
    └─ UserPromptPart: UserPromptPart(content="What is today's date and what year is it?", timestamp=datetime.datetime(2026, 5, 5, 21, 29, 54, 153452, tzinfo=datetime.timezone.utc))

[1] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='get_today', args='{}', tool_call_id='toolu_bdrk_011tvSiQKBxNzvG5btc1oz1J')

[2] ModelRequest
    └─ ToolReturnPart: ToolReturnPart(tool_name='get_today', content='2026-05-05', tool_call_id='toolu_bdrk_011tvSiQKBxNzvG5btc1oz1J', timestamp=datetime.datetime(2026, 5, 5, 21, 29, 55, 241387, tzinfo=datetime.timezone.utc

[3] ModelResponse
    └─ TextPart: TextPart(content="Today's date is **May 5, 2026** (2026-05-05), and the year is **2026**.")


---
## §3  Tools

An agent's power scales with the tools it has access to. We'll add two:

- `pubmed_search(query, k)` — return up to `k` cached PubMed abstracts
- `run_python(code)` — execute Python (sandboxed via `subprocess`)

The LLM picks which tool(s) to call based on the question.

In [40]:
from pydantic_ai import Agent

research_agent = Agent(
    MODEL,
    system_prompt=(
        'You are a research assistant. Use pubmed_search to find papers '
        'and run_python for any calculations or sequence analysis.'
    ),
)

@research_agent.tool_plain
def pubmed_search(query: str, k: int = 3) -> list[dict]:
    """Search cached PubMed abstracts.
    
    Returns up to k abstracts as dicts with keys: pmid, title, year, abstract.
    """
    query_words = [w for w in query.lower().split() if len(w) > 2]
    scored = []
    for a in CACHED_ABSTRACTS:
        text = (a['title'] + ' ' + a['abstract']).lower()
        score = sum(1 for w in query_words if w in text)
        if score > 0:
            scored.append((score, a))
    scored.sort(key=lambda x: -x[0])
    return [a for _, a in scored[:k]]

@research_agent.tool_plain
def run_python(code: str) -> str:
    """Execute Python code (sandboxed) and return stdout + stderr.
    
    Use for calculations, statistics, sequence analysis, or any
    task that requires running code. Timeout: 10 seconds.
    """
    import subprocess, tempfile, os as _os
    with tempfile.NamedTemporaryFile(suffix='.py', mode='w', delete=False) as f:
        f.write(code)
        fname = f.name
    try:
        proc = subprocess.run(
            ['python3', fname],
            capture_output=True, text=True, timeout=10,
        )
        return (proc.stdout + proc.stderr).strip() or '(no output)'
    finally:
        _os.unlink(fname)

print('Tools registered:', [t.name for t in research_agent._function_toolset.tools.values()])

Tools registered: ['pubmed_search', 'run_python']


In [41]:
# Use both tools in one query
result = research_agent.run_sync(
    'Find papers on cardiac amyloidosis. What is the median publication year '
    'of the results? Use Python to compute it.'
)
print(result.output)

## Results

I found papers on cardiac amyloidosis from the PubMed search. Here's what I retrieved:

### Papers Found:
1. **Deep learning identifies novel cardiac amyloidosis biomarkers in plasma proteomics** (PMID: 38421567, Year: 2024)
   - Uses a transformer-based model on plasma proteomic profiles to identify biomarkers for transthyretin amyloidosis

2. **CRISPR-Cas9 screen reveals zebrafish heart regeneration regulators** (PMID: 39012245, Year: 2024)
   - Though this result mentions cardiac/heart research, it's focused on regeneration rather than amyloidosis specifically

### Median Publication Year: **2024**

Both papers in the search results were published in 2024, making the median publication year **2024.0**.


**TODO (solution).** Add a `count_words(text)` tool, then ask the agent to count words in an abstract.

In [42]:
@research_agent.tool_plain
def count_words(text: str) -> int:
    """Return the number of whitespace-separated words in a piece of text."""
    return len(text.split())

result = research_agent.run_sync(
    'Search for papers on CRISPR. How many words are in the abstract '
    'of the first result?'
)
print(result.output)

The first search result is a 2024 paper titled **"CRISPR-Cas9 screen reveals zebrafish heart regeneration regulators"** (PMID: 39012245).

The abstract contains **44 words**.


---
## §4  Structured results

Free-text output is hard to use programmatically. With **Pydantic** the agent returns a typed object that's validated, serialisable, and ready to pass downstream.

In [43]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

class PaperSummary(BaseModel):
    """Structured summary of a single paper abstract."""
    title: str
    organism: str | None = Field(
        default=None,
        description="The organism studied (e.g. 'Homo sapiens'), or None if not applicable.",
    )
    method: str = Field(description="The main method or technique used.")
    key_metric: str = Field(description="The headline result, with units.")

extractor = Agent(
    MODEL,
    output_type=PaperSummary,
    system_prompt='Extract structured fields from a research paper abstract.',
)

In [44]:
# Run the extractor on the first cached abstract
abstract = CACHED_ABSTRACTS[0]
text = f"Title: {abstract['title']}\n\nAbstract: {abstract['abstract']}"

result = extractor.run_sync(text)
print(result.output.model_dump_json(indent=2))

# Typed access
print(f"\nOrganism: {result.output.organism}")
print(f"Method:   {result.output.method}")

{
  "title": "Deep learning identifies novel cardiac amyloidosis biomarkers in plasma proteomics",
  "organism": "Homo sapiens",
  "method": "Transformer-based deep learning model trained on plasma proteomic profiles",
  "key_metric": "0.94 AUROC on held-out test set; 17 novel biomarker candidates identified with 3 validated in independent cohort"
}

Organism: Homo sapiens
Method:   Transformer-based deep learning model trained on plasma proteomic profiles


**TODO (solution).** Extend `PaperSummary` with a `pmid` field validated by an 8-digit regex.

In [45]:
from pydantic import BaseModel, Field

class PaperSummaryV2(BaseModel):
    pmid: str = Field(pattern=r'^\d{8}$', description='8-digit PubMed ID')
    title: str
    organism: str | None = None
    method: str
    key_metric: str

extractor_v2 = Agent(
    MODEL,
    output_type=PaperSummaryV2,
    system_prompt='Extract structured fields from a paper abstract, including the PMID.',
)

for abstract in CACHED_ABSTRACTS[:3]:
    text = (
        f"PMID: {abstract['pmid']}\n"
        f"Title: {abstract['title']}\n\n"
        f"Abstract: {abstract['abstract']}"
    )
    out = extractor_v2.run_sync(text).output
    print(f"{out.pmid}: {out.method} ({out.organism})")

38421567: Transformer-based deep learning model trained on plasma proteomic profiles (Homo sapiens)
39012245: Genome-wide CRISPR-Cas9 knockout screen (Danio rerio)
37889123: Shotgun metagenomic sequencing with random-forest classifier (Faecalibacterium prausnitzii)


---
## §5  Context

Three ways an agent gets context:

1. **Conversation history** — multi-turn within a session.
2. **Document context** — paste relevant text into the prompt.
3. **Retrieval (RAG)** — when context is too big to paste, retrieve relevant chunks first.

In [46]:
# (a) Conversation history
from pydantic_ai import Agent

chat_agent = Agent(MODEL)

first = chat_agent.run_sync(
    'My favourite organism is the African clawed frog.'
)
print('Turn 1:', first.output)

second = chat_agent.run_sync(
    'What was the organism I just mentioned?',
    message_history=first.all_messages(),
)
print('Turn 2:', second.output)

Turn 1: That's a great choice! The African clawed frog (*Xenopus laevis*) is fascinating for many reasons:

**Scientific significance:**
- It's been hugely important to developmental biology and embryology research
- Played a key role in early work on cell signaling and gene expression
- Still used extensively in labs today for studying vertebrate development

**Interesting biology:**
- Those distinctive claws on their hind feet are used for gripping and tearing food
- They're fully aquatic and excellent swimmers
- Can survive in various water conditions and are quite hardy
- They lay thousands of eggs and develop rapidly, making them ideal for research
- They have an unusual mating amplexus where males ride on females for hours or days

**Unique features:**
- Relatively large eggs that are easy to observe under a microscope
- Transparent embryos allow direct observation of development
- They've been used to test pregnancy—their eggs responded to hormones in urine

Is there a particula

In [47]:
# (b) Document context — paste it into the prompt
abstract = CACHED_ABSTRACTS[1]   # the CRISPR/zebrafish paper

result = chat_agent.run_sync(
    f"Here is a paper abstract:\n\n{abstract['abstract']}\n\n"
    f"What organism was studied and what was the main finding?"
)
print(result.output)

# Study Summary

**Organism studied:** *Danio rerio* (zebrafish)

**Main finding:** The researchers identified 47 genes required for cardiac regeneration after ventricular resection through a genome-wide CRISPR-Cas9 knockout screen. Most notably, loss of the *nrg1* gene significantly impaired regeneration, reducing it by 68%.


In [48]:
# (c) Mini-RAG — semantic search over cached abstracts
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
abstract_texts = [a['title'] + '. ' + a['abstract'] for a in CACHED_ABSTRACTS]
abstract_embeddings = embedder.encode(abstract_texts, convert_to_numpy=True)

rag_agent = Agent(
    MODEL,
    system_prompt=(
        'Use search_abstracts to find relevant papers, then answer based on them.'
    ),
)

@rag_agent.tool_plain
def search_abstracts(query: str, k: int = 2) -> list[dict]:
    """Semantic search over cached PubMed abstracts. Returns up to k matches."""
    q_emb = embedder.encode([query], convert_to_numpy=True)[0]
    norms = np.linalg.norm(abstract_embeddings, axis=1) * np.linalg.norm(q_emb)
    scores = (abstract_embeddings @ q_emb) / norms
    top_idx = np.argsort(-scores)[:k]
    return [CACHED_ABSTRACTS[i] for i in top_idx]

result = rag_agent.run_sync(
    'What papers do we have on protein structure prediction?'
)
print(result.output)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Based on the search results, I found one highly relevant paper on protein structure prediction:

**1. AlphaFold-Multimer with diffusion refinement improves complex prediction (2025)**
   - PMID: 39765432
   - This paper extends AlphaFold-Multimer with a diffusion-based structural refinement module trained on 14,200 cryo-EM complexes. On the CASP16 benchmark, their method improved DockQ score by 18% over the baseline, with particularly impressive results for antibody-antigen complexes (32% DockQ gain).

The other papers in the results focused on different topics (biomarker discovery, microbiome analysis, genomics screening, and single-cell sequencing) rather than protein structure prediction.

Would you like me to search for more specific aspects of protein structure prediction, such as:
- AlphaFold or other specific methods
- Structure prediction combined with other techniques
- Protein folding or protein design
- Specific protein types (antibodies, enzymes, etc.)?


**TODO (solution).** Continue the conversation with a follow-up that depends on remembering both prior turns.

In [49]:
third = chat_agent.run_sync(
    'What is the scientific name of that organism?',
    message_history=second.all_messages(),
)
print('Turn 3:', third.output)

Turn 3: The scientific name is *Xenopus laevis*.


---
## §6  Multi-agent — orchestrator + workers

For complex tasks, decompose into specialised agents. The most common production pattern is **orchestrator-worker**: one component (an agent or just code) coordinates two or more specialists.

We'll refactor the literature pipeline into:

- **Extractor** — produces a `PaperSummary` (typed) from an abstract.
- **Summariser** — turns the structured fields into one English sentence.
- **Orchestrator** — a function that wires them together.

In [50]:
# The two specialist agents
from pydantic_ai import Agent

extractor = Agent(
    MODEL,
    output_type=PaperSummary,
    system_prompt='Extract structured fields from a paper abstract.',
)

summariser = Agent(
    MODEL,
    system_prompt=(
        'You are given structured fields about a paper. '
        'Write ONE plain-English sentence summarising the work.'
    ),
)

def process_abstract(text: str) -> dict:
    """Orchestrator: extract fields, then summarise them."""
    fields = extractor.run_sync(text).output
    sentence = summariser.run_sync(fields.model_dump_json()).output
    return {'fields': fields, 'summary': sentence}

In [51]:
# Run the pipeline on one abstract
abstract = CACHED_ABSTRACTS[0]
text = f"{abstract['title']}\n\n{abstract['abstract']}"

result = process_abstract(text)
print('Structured fields:')
print(result['fields'].model_dump_json(indent=2))
print(f"\nOne-line summary: {result['summary']}")

Structured fields:
{
  "title": "Deep learning identifies novel cardiac amyloidosis biomarkers in plasma proteomics",
  "organism": "Homo sapiens",
  "method": "Transformer-based deep learning model trained on plasma proteomic profiles",
  "key_metric": "0.94 AUROC on held-out test set; 17 novel biomarker candidates identified with 3 validated in independent cohort (n=412)"
}

One-line summary: A transformer-based deep learning model applied to plasma proteomics identified 17 novel biomarkers for cardiac amyloidosis with high diagnostic accuracy (0.94 AUROC), of which 3 were successfully validated in an independent cohort of 412 patients.


**TODO (solution).** Process all 5 cached abstracts in parallel using `asyncio.gather`.

In [52]:
import asyncio

async def process_async(text: str) -> dict:
    fields = (await extractor.run(text)).output
    sentence = (await summariser.run(fields.model_dump_json())).output
    return {'fields': fields, 'summary': sentence}

tasks = [
    process_async(f"{a['title']}\n\n{a['abstract']}")
    for a in CACHED_ABSTRACTS
]
results = await asyncio.gather(*tasks)

for r in results:
    print('•', r['summary'])

• A transformer-based deep learning model analyzed human plasma proteins to identify 17 new biomarkers for cardiac amyloidosis, achieving 94% accuracy with 3 candidates confirmed in independent patient samples.
• A genome-wide CRISPR-Cas9 knockout screen in zebrafish identified 47 genes required for heart regeneration, including nrg1, whose loss severely impaired regenerative capacity.
• Researchers used machine learning on stool microbiome data to predict which inflammatory bowel disease patients would relapse within six months with 81% accuracy.
• Single-cell RNA sequencing of nearly 190,000 cells from 41 non-small-cell lung cancers identified 22 distinct immune and stromal cell populations, revealing that higher frequencies of exhausted T cells predict better response to anti-PD-1 immunotherapy.
• AlphaFold-Multimer augmented with a diffusion-based refinement module trained on cryo-EM data improves protein complex structure prediction by 18% on the CASP16 benchmark, with particularl

---
## §7  Bonus — Real MCP in the wild

*Covered live if we have time, otherwise yours to run on your own.*

So far every tool was hand-written. **MCP (Model Context Protocol)** is a standard for sharing tools across teams and agents — instead of writing a `pubmed_search` function inside the notebook, you connect to a *server* that someone else maintains and your agent auto-loads its tools.

Below are five recipes for connecting to MCP servers in the wild. Each cell is self-contained — pick the ones relevant to your work and run those. **The agent code is identical across all of them**; only the `MCPServerStdio(command, args)` line changes.

### Recipe 1 — Python servers via `pip install` (BioMCP)

**BioMCP** (genomoncology/biomcp) exposes PubMed, ClinicalTrials.gov, and genomic-variant tools — actively maintained, MIT-licensed, ~20+ tools. Once you `pip install biomcp-python`, the agent below is functionally a biomedical research assistant: ask it about a disease and it'll search literature, find trials, and cross-reference variants without any biomedical-API glue code on your side.

In [53]:
# Step 1 — install BioMCP into the same venv (one-time, ~30s)
%pip install --quiet biomcp-python


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [54]:
# Step 2 — connect to BioMCP. Note how little code changed:
#   • Command:  'biomcp' instead of sys.executable
#   • Args:     ['run', '--mode', 'stdio'] instead of our stub script
# Everything else (Agent construction, run_mcp_servers, list_tools,
# the agent.run() call) is identical to our stub example.
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPServerStdio

biomcp_server = MCPServerStdio('biomcp', ['run', '--mode', 'stdio'])

bio_agent = Agent(
    MODEL,
    mcp_servers=[biomcp_server],
    system_prompt=(
        'You are a biomedical research assistant. Use the BioMCP tools '
        'to search PubMed, query ClinicalTrials.gov, and look up '
        'genomic variants. Cite tool results in your answer.'
    ),
)

async with bio_agent.run_mcp_servers():
    # Step 3 — see what BioMCP brings to the table:
    tools = await biomcp_server.list_tools()
    print(f'BioMCP exposes {len(tools)} tools. First 8:')
    for tool in tools[:8]:
        first_line = (tool.description or '').splitlines()[0][:90]
        print(f'  • {tool.name}: {first_line}')
    print()
    
    # Step 4 — ask a real biomedical question that needs real data:
    result = await bio_agent.run(
        'Find 3 recent (2024+) papers AND active clinical trials on '
        'transthyretin cardiac amyloidosis. Summarise the most '
        'promising therapies currently under investigation.'
    )
    print('--- Agent answer ---')
    print(result.output)

BioMCP exposes 36 tools. First 8:
  • search: Search biomedical literature, clinical trials, genetic variants, genes, drugs, and disease
  • fetch: Fetch comprehensive details for a specific biomedical record.
  • think: REQUIRED FIRST STEP: Perform structured sequential thinking for ANY biomedical research ta
  • article_searcher: Search PubMed/PubTator3 for research articles and preprints.
  • article_getter: Fetch detailed information for a specific article.
  • trial_searcher: Search ClinicalTrials.gov for clinical studies.
  • trial_getter: Fetch comprehensive details for a specific clinical trial.
  • trial_protocol_getter: Fetch core protocol information for a clinical trial.

--- Agent answer ---
Perfect! Now I have comprehensive information. Let me compile the findings for you.

## Summary: Recent Papers, Active Clinical Trials, and Promising Therapies for Transthyretin Cardiac Amyloidosis

### **3 Recent Papers (2024+) on ATTR-CM:**

1. **"Transthyretin cardiac amyloidosis: a

What just happened: a few lines of `MCPServerStdio(...)` config gave the agent access to PubMed + ClinicalTrials.gov + genomic-variant tools. That's the M$\times$N collapse, made tangible.

The same recipe works for every other MCP server. The flow is always: *install → find the launch command → point `MCPServerStdio` at it → use the agent normally*.

---
### Recipe pattern 2 — Python servers via `uvx` (no permanent install)

**`uvx`** (part of `uv`) runs a CLI tool from PyPI without putting it in your venv permanently. Great for one-off uses or for keeping your venv clean.

**Example: `mcp-server-fetch`** — an official server that lets the agent fetch any URL and convert it to LLM-friendly markdown. Ideal general-purpose tool: combine it with anything else.

In [68]:
# Make sure uv is installed first (one-time, outside the notebook):
#     curl -LsSf https://astral.sh/uv/install.sh | sh
# Then this cell will work.
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPServerStdio

fetch_server = MCPServerStdio('uvx', ['mcp-server-fetch'])

fetch_agent = Agent(
    MODEL,
    mcp_servers=[fetch_server],
    system_prompt='You can fetch any URL. Use the fetch tool when asked about online content.',
)

async with fetch_agent.run_mcp_servers():
    tools = await fetch_server.list_tools()
    print('Fetch server tools:', [t.name for t in tools])
    
    result = await fetch_agent.run(
        'Fetch https://en.wikipedia.org/wiki/Cardiac_amyloidosis and '
        'tell me in 2 sentences what subtypes of cardiac amyloidosis are most common.'
    )
    print(result.output)

Fetch server tools: ['fetch']
Based on the Wikipedia article on cardiac amyloidosis, the two most common subtypes are **transthyretin (TTR) amyloidosis**, which accounts for the majority of cases with 75% being wild-type and 25% being the hereditary form, and **light chain (AL) amyloidosis**, which is one of the most studied types. The TTR form is more common in older adults (mean age of diagnosis 74-90), while AL amyloidosis typically affects males over age 60 and is more rapidly progressive.


---
### Recipe pattern 3 — servers that need authentication (env vars)

Servers that talk to private services (GitHub, Slack, Notion, internal databases) need credentials. The MCP convention is **environment variables**, set just before launching the subprocess via `MCPServerStdio(..., env={...})`.

**Example: GitHub MCP** — repos, issues, pull requests. Needs a GitHub Personal Access Token (PAT). Generate one at github.com/settings/tokens/new with the `repo` scope, then:

In [ ]:
# Don't run this unless you actually want to test it — it requires a
# real GitHub PAT. Set it in your shell as GITHUB_TOKEN, or paste
# directly here (don't commit a real token to a notebook).
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')   # or 'ghp_...' literal

if GITHUB_TOKEN:
    github_server = MCPServerStdio(
        'npx',
        ['-y', '@modelcontextprotocol/server-github'],
        env={
            **os.environ,
            'GITHUB_PERSONAL_ACCESS_TOKEN': GITHUB_TOKEN,
        },
    )
    
    github_agent = Agent(
        MODEL,
        mcp_servers=[github_server],
        system_prompt='You can browse GitHub repos, issues, and PRs.',
    )
    
    async with github_agent.run_mcp_servers():
        result = await github_agent.run(
            'How many open issues does the modelcontextprotocol/servers '
            'repository have right now?'
        )
        print(result.output)
else:
    print('Skipping — set GITHUB_TOKEN env var to try this cell.')

---
### Recipe pattern 4 — community servers that need cloning

Some bio-specific MCP servers aren't packaged on PyPI/npm yet — you clone the repo, set up a venv, and point `MCPServerStdio` at the Python interpreter inside that venv. **The MCP code in your notebook is unchanged**; only the *launch command* changes.

In [ ]:
# Example template — UniProt MCP (Augmented-Nature/Augmented-Nature-UniProt-MCP-Server)
# This server is Node-based; clone, npm install, then point MCPServerStdio at the build.
#
#  Outside the notebook (one-time, in a terminal):
#    git clone https://github.com/Augmented-Nature/Augmented-Nature-UniProt-MCP-Server.git
#    cd Augmented-Nature-UniProt-MCP-Server
#    npm install
#    npm run build
#
#  Then in the notebook (replace the path with your actual checkout):
UNIPROT_PATH = '/PATH/TO/Augmented-Nature-UniProt-MCP-Server/build/index.js'

EXAMPLE_UNIPROT_USAGE = '''
uniprot_server = MCPServerStdio('node', [UNIPROT_PATH])

uniprot_agent = Agent(
    MODEL,
    mcp_servers=[uniprot_server],
    system_prompt='You can search the UniProt protein database.',
)

async with uniprot_agent.run_mcp_servers():
    result = await uniprot_agent.run(
        'Find UniProt entries for the human transthyretin protein '
        'and list its top three known disease associations.'
    )
    print(result.output)
'''
print(EXAMPLE_UNIPROT_USAGE)
print('(template only — set UNIPROT_PATH and uncomment to actually run)')

---
### Cheat-sheet — every recipe in one place

| Server | Install pattern | `MCPServerStdio(command, args)` |
|---|---|---|
| **BioMCP** | `pip install biomcp-python` | `('biomcp', ['run', '--mode', 'stdio'])` |
| **Fetch** | `uvx mcp-server-fetch` | `('uvx', ['mcp-server-fetch'])` |
| **Filesystem** | `npx -y @modelcontextprotocol/server-filesystem` | `('npx', ['-y', '@modelcontextprotocol/server-filesystem', './'])` |
| **GitHub** | `npx -y @modelcontextprotocol/server-github` + `GITHUB_PERSONAL_ACCESS_TOKEN` | `('npx', ['-y', '@modelcontextprotocol/server-github'], env={...})` |
| **UniProt** (Augmented-Nature) | clone + `npm install && npm run build` | `('node', ['/path/to/build/index.js'])` |
| **PubMed** (aeghnnsw) | clone + `bash setup.sh` | `('/path/to/venv/bin/python', ['/path/to/server.py'])` |
| **AlphaFold MCP** (Augmented-Nature) | clone + `npm install && npm run build` | `('node', ['/path/to/build/index.js'])` |
| **PDB MCP** (Augmented-Nature) | clone + `npm install && npm run build` | `('node', ['/path/to/build/index.js'])` |
| **ChEMBL MCP** (Augmented-Nature) | clone + `npm install && npm run build` | `('node', ['/path/to/build/index.js'])` |

**The pattern**

The `MCPServerStdio(command, args, env=...)` call is the *only* thing that changes between servers. Everything else — `Agent` construction, `run_mcp_servers()`, `list_tools()`, the agent's `run()` call — is identical. Once you know the pattern, you can wire any of the 9{,}400+ registered MCP servers into your agent in under a minute.

**Repos & references**

- BioMCP — github.com/genomoncology/biomcp
- Official MCP servers — github.com/modelcontextprotocol/servers
- Augmented-Nature bio servers (UniProt, AlphaFold, PDB, ChEMBL, OpenTargets) — github.com/Augmented-Nature
- PubMed MCP — github.com/aeghnnsw/pubmed-mcp
- MCPmed initiative — *Briefings in Bioinformatics* (2026): academic.oup.com/bib/article/27/1/bbag076
- MCP server registry — github.com/modelcontextprotocol/registry

---
## §8  Bonus — Reflection

*Covered live if we have time, otherwise yours to run on your own.*

**Reflection** is the design pattern where an agent critiques its own output and produces an improved version. We'll wire a *critic agent* next to the extractor from §4: the critic reads the original abstract plus the extracted fields and either approves them or flags issues.

The trick that makes reflection work: give the critic concrete, checkable criteria, not vague prompts like *'is this good?'*

In [55]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

class CriticVerdict(BaseModel):
    """The critic's assessment of an extraction."""
    approved: bool = Field(description='True if the extraction is acceptable.')
    issues: list[str] = Field(
        default_factory=list,
        description='Specific problems found, if any. Empty if approved.',
    )

critic = Agent(
    MODEL,
    output_type=CriticVerdict,
    system_prompt=(
        'You check structured extractions against the original abstract. '
        'Flag issues like: empty fields, organism missing when one is studied, '
        'method that does not match the abstract, key_metric without units.'
    ),
)

In [56]:
# Extract → critique → optionally re-extract
abstract = CACHED_ABSTRACTS[0]
text = f"Title: {abstract['title']}\n\nAbstract: {abstract['abstract']}"

v1 = extractor.run_sync(text).output
print('Extraction v1:')
print(v1.model_dump_json(indent=2))

verdict = critic.run_sync(
    f'Original abstract:\n{text}\n\n'
    f'Extracted fields:\n{v1.model_dump_json()}\n\n'
    f'Are these fields acceptable?'
).output
print('\nCritic verdict:', verdict)

Extraction v1:
{
  "title": "Deep learning identifies novel cardiac amyloidosis biomarkers in plasma proteomics",
  "organism": "Homo sapiens",
  "method": "Transformer-based deep learning model trained on plasma proteomic profiles",
  "key_metric": "0.94 AUROC on held-out test set; 17 novel biomarker candidates identified with 3 validated in independent cohort"
}

Critic verdict: approved=True issues=[]


**TODO (solution).** Build a `reflect_until_approved` function that loops up to 2 retries: extract, critique, and only re-extract if the critic flags issues.

In [57]:
def reflect_until_approved(abstract_text: str, max_retries: int = 2):
    """Extract; if the critic flags issues, re-extract with feedback."""
    history = []
    candidate = extractor.run_sync(abstract_text).output
    
    for attempt in range(max_retries):
        verdict = critic.run_sync(
            f'Original abstract:\n{abstract_text}\n\n'
            f'Extracted fields:\n{candidate.model_dump_json()}\n\n'
            f'Are these fields acceptable?'
        ).output
        history.append((candidate, verdict))
        if verdict.approved:
            break
        # Re-extract with the critic's feedback as context
        candidate = extractor.run_sync(
            f'{abstract_text}\n\n'
            f'Previous attempt had these issues: {verdict.issues}. '
            f'Please re-extract carefully.'
        ).output
    return candidate, history

final, history = reflect_until_approved(text)
print(f'Final extraction (after {len(history)} critique round(s)):')
print(final.model_dump_json(indent=2))

Final extraction (after 1 critique round(s)):
{
  "title": "Deep learning identifies novel cardiac amyloidosis biomarkers in plasma proteomics",
  "organism": "Homo sapiens",
  "method": "Transformer-based deep learning model trained on plasma proteomic profiles",
  "key_metric": "0.94 AUROC on held-out test set; 17 novel biomarker candidates identified"
}


---
## §9  Bonus — Reasoning models

*Covered live if we have time, otherwise yours to run on your own.*

**Reasoning models** (Claude with extended thinking, OpenAI o-series, Gemini Deep Think, DeepSeek-R1) pause and *think* before answering. On hard problems — multi-step math, debugging, plan generation — this consistently outperforms a non-reasoning model of comparable size. On simple knowledge-recall tasks they sometimes do **worse** because they overthink.

On OpenRouter the switch is one line: change the model slug. The agent code below stays identical.

In [58]:
# Build a reasoning-model variant of the extractor.
# Try several reasoning slugs available on OpenRouter:
#   • 'openai/o3-mini'
#   • 'openai/gpt-5'              (with reasoning_effort='medium')
#   • 'anthropic/claude-sonnet-4.5'  (with extended thinking)
#   • 'deepseek/deepseek-r1'
#
# Pick whatever your OpenRouter account has access to.
REASONING_SLUG = 'openai/o3-mini'

REASONING_MODEL = OpenAIChatModel(
    REASONING_SLUG,
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=HTTPX_CLIENT,   # share the safe client (see setup §3)
    ),
)

reasoning_extractor = Agent(
    REASONING_MODEL,
    output_type=PaperSummary,
    system_prompt='Extract structured fields from a paper abstract. Reason carefully about the method and the key metric.',
)

print(f'✓ Reasoning model configured: {REASONING_SLUG}')

✓ Reasoning model configured: openai/o3-mini


In [59]:
# Run both models on the same abstract and compare.
# A deliberately tricky case: the cached AlphaFold paper (multiple methods, multiple metrics).
import time

abstract = CACHED_ABSTRACTS[4]   # AlphaFold-Multimer paper
text = f"Title: {abstract['title']}\n\nAbstract: {abstract['abstract']}"

t0 = time.time()
fast_result = extractor.run_sync(text).output
fast_dt = time.time() - t0

t0 = time.time()
slow_result = reasoning_extractor.run_sync(text).output
slow_dt = time.time() - t0

print(f'Fast model ({MODEL_SLUG}) — {fast_dt:.1f}s')
print(fast_result.model_dump_json(indent=2))
print()
print(f'Reasoning model ({REASONING_SLUG}) — {slow_dt:.1f}s')
print(slow_result.model_dump_json(indent=2))

Fast model (anthropic/claude-haiku-4.5) — 2.0s
{
  "title": "AlphaFold-Multimer with diffusion refinement improves complex prediction",
  "organism": "Homo sapiens",
  "method": "AlphaFold-Multimer extended with diffusion-based structural refinement module trained on 14,200 cryo-EM complexes",
  "key_metric": "18% improvement in DockQ score over baseline on CASP16 benchmark (32% for antibody-antigen complexes)"
}

Reasoning model (openai/o3-mini) — 4.1s
{
  "title": "AlphaFold-Multimer with diffusion refinement improves complex prediction",
  "organism": null,
  "method": "Extended AlphaFold-Multimer with a diffusion-based structural refinement module trained on 14,200 cryo-EM complexes",
  "key_metric": "DockQ score improvement of 18% overall, with up to a 32% gain for antibody-antigen complexes"
}


**What to look for.** Reasoning models tend to:
- Pick the most *specific* method (e.g.\ "diffusion-based structural refinement" vs.\ "deep learning").
- Get the right *headline* metric when several are mentioned.
- Produce more carefully scoped `organism` values.

They also take 5–30× longer. Default to the fast model; route to a reasoning model only on the steps where quality matters more than latency.

---
## §10  Bonus — Evals

*Covered live if we have time, otherwise yours to run on your own.*

**Evals** are the discipline that turns vibes ("this seems fine") into numbers ("this is 85% correct on 20 examples"). Without them you can't tell whether a change actually improved the system.

We'll build three evals against the extractor:

1. **Code + per-example ground truth** — exact-match on `organism`.
2. **Code + shared rule** — every `key_metric` must contain a digit.
3. **LLM-as-judge** — a separate model rates summary quality 0–3.

In [60]:
# Ground-truth labels for our 5 cached abstracts.
GROUND_TRUTH = {
    '38421567': {'organism': 'Homo sapiens'},
    '39012245': {'organism': 'Danio rerio'},
    '37889123': {'organism': 'Homo sapiens'},
    '40123987': {'organism': 'Homo sapiens'},
    '39765432': {'organism': None},   # method paper, no specific organism
}

# Run the extractor on every cached abstract, store outputs.
extractions = {}
for a in CACHED_ABSTRACTS:
    text = f"Title: {a['title']}\n\nAbstract: {a['abstract']}"
    extractions[a['pmid']] = extractor.run_sync(text).output

print(f'Extracted {len(extractions)} papers.')

Extracted 5 papers.


In [61]:
# Eval 1 — code + per-example ground truth (exact match on organism)
def eval_organism_match(extractions, truth):
    correct = 0
    for pmid, expected in truth.items():
        actual = extractions[pmid].organism
        if expected['organism'] is None:
            ok = actual is None or actual.lower() in {'none', 'n/a'}
        else:
            ok = actual is not None and expected['organism'].lower() in actual.lower()
        if ok: correct += 1
        print(f"  {pmid}: expected={expected['organism']!r} actual={actual!r} {'✓' if ok else '✗'}")
    return correct / len(truth)

score_1 = eval_organism_match(extractions, GROUND_TRUTH)
print(f'\nEval 1 (organism exact-match): {score_1:.0%}')

  38421567: expected='Homo sapiens' actual='Homo sapiens' ✓
  39012245: expected='Danio rerio' actual='Danio rerio' ✓
  37889123: expected='Homo sapiens' actual='Homo sapiens' ✓
  40123987: expected='Homo sapiens' actual='Homo sapiens' ✓
  39765432: expected=None actual='Homo sapiens' ✗

Eval 1 (organism exact-match): 80%


In [62]:
# Eval 2 — code + shared rule (every key_metric must contain a digit)
import re

def eval_metric_has_digit(extractions):
    correct = sum(1 for x in extractions.values() if re.search(r'\d', x.key_metric))
    return correct / len(extractions)

score_2 = eval_metric_has_digit(extractions)
print(f'Eval 2 (key_metric contains a digit): {score_2:.0%}')

Eval 2 (key_metric contains a digit): 100%


In [63]:
# Eval 3 — LLM-as-judge with a binary rubric
from pydantic import BaseModel, Field

class JudgeScore(BaseModel):
    has_clear_method: bool
    has_specific_metric: bool
    organism_correctly_identified: bool

judge = Agent(
    MODEL,
    output_type=JudgeScore,
    system_prompt=(
        'You are an evaluator. Given a paper abstract and an extracted '
        'PaperSummary, score each criterion as True or False.'
    ),
)

def eval_llm_judge(extractions):
    total_points = 0
    max_points = 0
    for a in CACHED_ABSTRACTS:
        extraction = extractions[a['pmid']]
        prompt = (
            f"Abstract: {a['abstract']}\n\n"
            f"Extracted: {extraction.model_dump_json()}\n"
        )
        score = judge.run_sync(prompt).output
        points = sum([score.has_clear_method, score.has_specific_metric, score.organism_correctly_identified])
        total_points += points
        max_points += 3
        print(f"  {a['pmid']}: {points}/3")
    return total_points / max_points

score_3 = eval_llm_judge(extractions)
print(f'\nEval 3 (LLM-judge rubric): {score_3:.0%}')

  38421567: 3/3
  39012245: 3/3
  37889123: 3/3
  40123987: 3/3
  39765432: 2/3

Eval 3 (LLM-judge rubric): 93%


### Why this matters

Even with 5 examples, you now have three independent quality signals. When you tweak the prompt or swap the model, you re-run these evals and *measure* whether the change helped — instead of guessing.

In production: scale to 50–200 examples, plug into LangSmith / Logfire / Langfuse for tracking over time, and treat the evals as gating tests (don't ship a change that drops Eval 1 below some threshold).

---
## §11  Bonus — Memory

*Covered live if we have time, otherwise yours to run on your own.*

**Conversation history** (§5a) gives the agent context within a session. **Memory** is what survives between sessions: facts the agent should remember next time, even after the kernel restarts.

The simplest possible memory is a JSON file — and it's surprisingly competitive. Production frameworks (Letta, Mem0, Zep) layer vectorisation, temporal reasoning, and graph relations on top of the same idea.

In [64]:
# A tiny file-based memory store, exposed to the agent as two tools.
import json
from pathlib import Path

MEMORY_FILE = Path('agent_memory.json')

def _load_memory() -> dict:
    if MEMORY_FILE.exists():
        return json.loads(MEMORY_FILE.read_text())
    return {}

def _save_memory(mem: dict) -> None:
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

memory_agent = Agent(
    MODEL,
    system_prompt=(
        'You are a research assistant with persistent memory. '
        'Use remember() to save facts the user wants kept, and '
        'recall_all() to read everything you have saved.'
    ),
)

@memory_agent.tool_plain
def remember(key: str, value: str) -> str:
    """Save a key/value fact to long-term memory."""
    mem = _load_memory()
    mem[key] = value
    _save_memory(mem)
    return f'Saved: {key}'

@memory_agent.tool_plain
def recall_all() -> dict:
    """Return every key/value pair currently in long-term memory."""
    return _load_memory()

# Wipe any prior memory for this demo run
if MEMORY_FILE.exists():
    MEMORY_FILE.unlink()
print('✓ Memory tools ready, store cleared.')

✓ Memory tools ready, store cleared.


In [65]:
# Save a few facts
memory_agent.run_sync(
    "Remember that I'm interested in cardiac amyloidosis and zebrafish models."
)
memory_agent.run_sync(
    "Also remember my preferred citation style is Vancouver."
)

print('Stored memory:', _load_memory())

Stored memory: {'research_interests': 'cardiac amyloidosis and zebrafish models', 'preferred_citation_style': 'Vancouver'}


In [66]:
# In a 'new session' (the kernel survives, but agent.run_sync starts a fresh
# conversation each call) — the agent can still recall what it saved.
result = memory_agent.run_sync(
    'What topics am I interested in, and what citation style do I prefer?'
)
print(result.output)

Based on my memory, here's what I have about you:

**Research Interests:** Cardiac amyloidosis and zebrafish models

**Preferred Citation Style:** Vancouver

Would you like to update any of this information or add additional details?


**TODO (solution).** Add a `forget(key)` tool, then ask the agent to forget the citation-style preference and confirm it's gone.

In [67]:
@memory_agent.tool_plain
def forget(key: str) -> str:
    """Remove a key from long-term memory."""
    mem = _load_memory()
    if key in mem:
        del mem[key]
        _save_memory(mem)
        return f'Forgot: {key}'
    return f'No such key: {key}'

memory_agent.run_sync('Please forget my preferred citation style.')
result = memory_agent.run_sync('What do you currently remember about me?')
print(result.output)
print('\nRaw memory file:', _load_memory())

I currently remember one thing about you:

**Research interests:** Cardiac amyloidosis and zebrafish models

Is there anything else you'd like me to remember or update about you?

Raw memory file: {'research_interests': 'cardiac amyloidosis and zebrafish models'}


---
## Where to go next

This 1-hour tutorial covered six core building blocks (§1–§6), plus five bonus sections (§7–§11) that show how the same primitives compose into more sophisticated patterns.

**Other topics worth exploring:**

- **Computer use & voice** — agents that drive GUIs and speak (Module 6).
- **Safety** — prompt injection, capability scoping (Module 7).
- **Observability at scale** — Pydantic Logfire, LangSmith, Langfuse.
- **Multi-agent in production** — the orchestrator-worker pattern (Module 5.6).

**One question to leave with.** What's the smallest agent you could build this week that would save you (or a colleague) real time? Pick that, build it, and instrument it with evals and observability from day one.